## 0. Configuración de rutas

In [ ]:
import os

# El notebook vive en notebooks/ — subimos un nivel para llegar a la raiz del proyecto
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")

PROC_CC = os.path.join(DATOS_PATH, "processed", "cc_news")
PROC_MIND = os.path.join(DATOS_PATH, "processed", "mind_large")
TFIDF_CC = os.path.join(DATOS_PATH, "processed", "tfidf_cc")
TFIDF_MIND = os.path.join(DATOS_PATH, "processed", "tfidf_mind")
INDEX_PATH = os.path.join(DATOS_PATH, "processed", "inverted_index")
MODELS_PATH = os.path.join(PROJECT_PATH, "models")

for path in [TFIDF_CC, TFIDF_MIND, INDEX_PATH, MODELS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"PROJECT_PATH → {PROJECT_PATH}")
print("Rutas configuradas.")

## 1. Dependencias

In [ ]:
# Ejecuta solo la primera vez
# !pip install pyspark nltk

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

STOPWORDS_EN = set(stopwords.words("english"))
print(f"Stopwords cargadas: {len(STOPWORDS_EN)}")

## 2. Inicialización de PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

spark = (
    SparkSession.builder
    .appName("Preprocesamiento-MapReduce")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

## 3. Carga de datos (salida de Fase 1)

In [ ]:
cc_df = spark.read.parquet(PROC_CC)
mind_df = spark.read.parquet(PROC_MIND)

print(f"CC-News:    {cc_df.count():>8,} documentos")
print(f"MIND Large: {mind_df.count():>8,} documentos")

# Estandarizamos esquema: (doc_id, text, label)
# CC-News no tiene etiqueta -> None
cc_std = (
    cc_df
    .select(
        F.col("doc_id").cast("string"),
        F.col("text"),
        F.lit(None).cast("string").alias("label")
    )
)

# MIND tiene etiqueta de categoria
mind_std = (
    mind_df
    .select(
        F.col("news_id").alias("doc_id"),
        F.col("text"),
        F.col("category").alias("label")
    )
)

cc_std.show(2, truncate=80)
mind_std.show(2, truncate=80)

## 4. MAP — Tokenización

Cada documento se convierte en una lista de tokens en minúsculas, eliminando puntuación y números.

**Paralelo al modelo MapReduce:** esta transformación es independiente por documento → se ejecuta en cada partición sin comunicación entre nodos.

In [ ]:
import re

def tokenizar(texto):
    """Map: texto → lista de tokens crudos."""
    if not texto:
        return []
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]", " ", texto)  # solo letras
    tokens = texto.split()
    return [t for t in tokens if len(t) > 2]  # descarta tokens de 1-2 chars

tokenizar_udf = F.udf(tokenizar, ArrayType(StringType()))

# Aplicar Map a ambos datasets
cc_tokens   = cc_std.withColumn("tokens_raw",   tokenizar_udf(F.col("text")))
mind_tokens = mind_std.withColumn("tokens_raw", tokenizar_udf(F.col("text")))

print("Ejemplo de tokenización:")
mind_tokens.select("doc_id", "tokens_raw").show(3, truncate=80)

In [ ]:
from pyspark.ml import PipelineModel

# Guardar vectores TF-IDF
(cc_tfidf
    .select("doc_id", "tfidf_vector")
    .repartition(8)
    .write.mode("overwrite")
    .parquet(TFIDF_CC))
print(f"TF-IDF CC-News guardado: {TFIDF_CC}")

(mind_tfidf
    .select("doc_id", "label", "tfidf_vector")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(TFIDF_MIND))
print(f"TF-IDF MIND guardado:    {TFIDF_MIND}")

# Guardar modelo IDF de MIND (necesario para inferencia en Fase 4)
idf_model_mind.save(os.path.join(MODELS_PATH, "idf_model_mind"))
print("Modelo IDF guardado.")

## 6. REDUCE — TF-IDF con MLlib

**HashingTF** calcula la frecuencia de términos (TF) por documento usando hashing trick.  
**IDF** agrega el factor inverso de frecuencia de documentos — requiere una pasada global (shuffle) → aquí está el costo de comunicación del Reduce.

```
TF(t, d)  = frecuencia del término t en documento d
IDF(t)    = log( N / df(t) )   donde N = total docs, df(t) = docs que contienen t
TF-IDF    = TF * IDF
```

In [ ]:
from pyspark.ml.feature import HashingTF, IDF
from pyspark.ml import Pipeline

NUM_FEATURES = 65536  # 2^16 — tamanio del espacio de caracteristicas

hashing_tf = HashingTF(inputCol="tokens", outputCol="tf_vector", numFeatures=NUM_FEATURES)
idf        = IDF(inputCol="tf_vector", outputCol="tfidf_vector", minDocFreq=3)

def calcular_tfidf(df, nombre):
    print(f"Calculando TF para {nombre}...")
    tf_df = hashing_tf.transform(df)

    print(f"Ajustando IDF para {nombre} (operación Reduce global — shuffle)...")
    idf_model = idf.fit(tf_df)
    tfidf_df  = idf_model.transform(tf_df)

    return tfidf_df.select("doc_id", "label", "tokens", "tfidf_vector"), idf_model

cc_tfidf,   idf_model_cc   = calcular_tfidf(cc_limpio,   "CC-News")
mind_tfidf, idf_model_mind = calcular_tfidf(mind_limpio, "MIND")

print("\nEjemplo de vector TF-IDF (primeros 5 valores distintos de cero):")
mind_tfidf.select("doc_id", "tfidf_vector").limit(2).show(truncate=80)

## 7. Guardado de vectores TF-IDF

In [ ]:
from pyspark.ml import PipelineModel

# Guardar vectores TF-IDF
(cc_tfidf
    .select("doc_id", "tfidf_vector")
    .repartition(8)
    .write.mode("overwrite")
    .parquet(TFIDF_CC))
print(f"TF-IDF CC-News guardado: {TFIDF_CC}")

(mind_tfidf
    .select("doc_id", "label", "tfidf_vector")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(TFIDF_MIND))
print(f"TF-IDF MIND guardado:    {TFIDF_MIND}")

# Guardar modelo IDF de MIND (necesario para inferencia en Fase 4)
idf_model_mind.save(os.path.join(BASE_PATH, "models", "idf_model_mind"))
print("Modelo IDF guardado.")

## 8. Índice invertido (MapReduce explícito con RDDs)

Construcción manual con RDDs para mostrar el patrón MapReduce puro del curso.

```
Map:    (doc_id, tokens) → [(token, doc_id), ...]
Reduce: (token, [doc_id1, doc_id2, ...]) → índice invertido
```

In [ ]:
from pyspark.ml import PipelineModel

# Guardar vectores TF-IDF
(
    cc_tfidf
    .select("doc_id", "tfidf_vector")
    .repartition(8)
    .write.mode("overwrite")
    .parquet(TFIDF_CC)
)
print(f"TF-IDF CC-News guardado: {TFIDF_CC}")

(
    mind_tfidf
    .select("doc_id", "label", "tfidf_vector")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(TFIDF_MIND)
)
print(f"TF-IDF MIND guardado:    {TFIDF_MIND}")

# Guardar modelo IDF de MIND (necesario para inferencia en Fase 4)
idf_model_mind.save(os.path.join(MODELS_PATH, "idf_model_mind"))
print("Modelo IDF guardado.")

In [ ]:
# Guardar índice invertido como Parquet (token → lista de doc_ids)
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema_indice = StructType([
    StructField("token",   StringType(),             False),
    StructField("doc_ids", ArrayType(StringType()),  False),
])

indice_df = spark.createDataFrame(
    indice_invertido.map(lambda x: (x[0], x[1])),
    schema=schema_indice
)

indice_df.write.mode("overwrite").parquet(INDEX_PATH)
print(f"Índice invertido guardado: {INDEX_PATH}")
print(f"Términos únicos indexados: {indice_df.count():,}")

## 9. Análisis costo-comunicación — Fase 2

In [ ]:
n_docs  = mind_limpio.count()
n_terms = indice_df.count()

print("MODELO COSTO-COMUNICACIÓN — PREPROCESAMIENTO (Fase 2)")
print("-" * 55)
print(f"Documentos procesados : {n_docs:,}")
print(f"Términos únicos (vocab): {n_terms:,}")
print(f"Dimensión vector TF-IDF: {NUM_FEATURES:,} (hashing trick)")
print()
print("Costo por etapa:")
print(f"  Map (tokenizar):      O(n · L)   — L = longitud media del documento")
print(f"  Combiner (stem/stop): O(n · L)   — local, sin comunicación")
print(f"  Reduce TF:            O(n · L)   — local por partición")
print(f"  Reduce IDF (shuffle): O(|V| · p) — |V|=vocab, p=particiones (COSTOSO)")
print(f"  Índice invertido:     O(n · L)   — 1 shuffle groupByKey")
print()
print("Optimizaciones aplicadas:")
print("  - Broadcast de stopwords: evita transferir la lista a cada tarea")
print("  - minDocFreq=3 en IDF: elimina términos raros → reduce |V| efectivo")
print("  - HashingTF: evita diccionario global (sin comunicación en Map)")

In [ ]:
spark.stop()
print("SparkSession cerrada. Fase 2 completada.")
print("Siguiente paso: 03_lsh_dedup.ipynb")